# 未来のメロンの価格の予測

#### このノートのメインURL

データセット：https://www.kaggle.com/datasets/nagatakengo/kaggriculture-movements-in-the-top-xx

メインnotebook：https://www.kaggle.com/code/nagatakengo/kaggriculture-movements-top-xx


In [ ]:
#!unzip /content/df_all_data_final.zip

In [ ]:
import sys
sys.path.append("/kaggle/input/datasets/nagatakengo/kaggriculture-movements-in-the-top-xx")

import pandas as pd
df = pd.read_csv('/kaggle/input/datasets/nagatakengo/kaggriculture-movements-in-the-top-xx/df_all_data_final/df_all_data_final.csv')

In [ ]:
def df_cl(df):
    df_test = df [["id",
                   "step",
                   "day",
                   "money",
                   "hands_n",
                   "hires_today",
                   "unlocked_quadrants",
                   "tiles",
                   "private_shed",
                   "private_seeds",
                   "market_inv",
                   "market_prices",
                   "market_action"
                   ]].copy()
    return df_test

df_all = df_cl(df)

print(f"df_allの列: {df_all.columns}")
print("")
print(f"df_allの欠損値: {df_all.isnull().values.any()}")
print("")
print(f"df_allの行と列: {df_all.shape[0]}行  {df_all.shape[1]}列")




df_allの列: Index(['id', 'step', 'day', 'money', 'hands_n', 'hires_today',
       'unlocked_quadrants', 'tiles', 'private_shed', 'private_seeds',
       'market_inv', 'market_prices', 'market_action'],
      dtype='object')

df_allの欠損値: False

df_allの行と列: 214560行  13列


# DFの構造


```
df_allの列: Index(['id', 'step', 'day', 'money', 'hands_n', 'hires_today',
       'unlocked_quadrants', 'tiles', 'private_shed', 'private_seeds',
       'market_inv', 'market_prices', 'market_action'],
      dtype='object')


"id"=1 下から10%の実力の30名分の1試合ずつ (21600 rows  13 columns)スコア50000くらい
"id"=2 下から20%の実力の30名分の1試合ずつ (21600 rows  13 columns)スコア60000くらい
"id"=3 下から30%の実力の30名分の1試合ずつ (21600 rows  13 columns)スコア69000くらい
"id"=4 下から40%の実力の30名分の1試合ずつ (21600 rows  13 columns)スコア76000くらい
"id"=5 下から50%の実力の30名分の1試合ずつ (21600 rows  13 columns)スコア83000くらい
"id"=6 下から60%の実力の30名分の1試合ずつ (21600 rows  13 columns)スコア92000くらい
"id"=7 下から70%の実力の30名分の1試合ずつ (21600 rows  13 columns)スコア100000くらい
"id"=8 下から80%の実力の30名分の1試合ずつ (21600 rows  13 columns)スコア110000くらい
"id"=9 下から90%の実力の30名分の1試合ずつ (21600 rows  13 columns)スコア120000くらい
"id"=10 上位1%の実力の28名分の1試合ずつ （20160 rows  13 columns）スコア150000くらい


df_all:(全部繋げた)
214560 rows  13 columns


214,560行
合計298試合
1試合720行
```

# 学習のイメージ

``現在の情報 → 数十ターン先までのMELON最高価格を予測 → 上がり代が大きければHOLD、小さければSELL``



### 特徴量
```
"step": ステップ数
"current_price" : 現在のMELON価格
"market_stock" : MELON市場在庫
"Past_price_changes": 過去の価格変化
"Past_inv_changes": 過去の在庫変化
"Past_SELL": 過去のMELON SELL
"amount_in_shed": shedのMELON量

"future_max_price": 現在より先xターン以内のMELON最高価格(y)
```



* 「何ターン先の価格の上がり代を予測するか」を決めて、目的変数を作る。
* 中～上位勢が同じNotebookを使っているため、行をランダム分割して学習・評価しないこと
* 24ターン=1日なので、この3つは``約0.5日``, ``1日``, ``2日`` の予測比較になります

# 評価

**MAE（平均絶対誤差）**

平均で何ドル外したか?

$$MAE = \frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y}_i|$$

* $n$ ：データの総数
* $y_i$ ：実際の値（正解）
* $\hat{y}_i$ ：予測した値
* $| \quad |$ ：絶対値

<br>


**RMSE（平方根平均二乗誤差）**

大きな外しをどれだけ出したか?

$$RMSE = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2}$$

* $n$ ：データの総数
* $y_i$ ：実際の値（正解）
* $\hat{y}_i$ ：予測した値





# DF作成

In [ ]:
import ast


def df_cl(df, lookbacks=(12, 24, 48)):
    """
    df作成
    """
    df_res = df.copy()

    # 試合ID
    df_res["game_id"] = df_res["step"].eq(0).cumsum()


    current_price = df_res["market_prices"].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )

    market_stock = df_res["market_inv"].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )

    market_actions = df_res["market_action"].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )

    private_shed = df_res["private_shed"].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )

    # shed内のMELON量
    df_res["amount_in_shed"] = private_shed.apply(
        lambda x: x.get("MELON", 0)
    )

    # 現在のMELON価格
    df_res["current_price"] = current_price.str["MELON"]

    # MELON市場在庫
    df_res["market_stock"] = market_stock.str["MELON"]


    # 現在ターンのMELON SELL量
    def extract_melon_sell(actions):

        total = 0

        for order in actions:
            if (
                isinstance(order, (list, tuple))
                and len(order) >= 3
                and order[0] == "SELL"
                and order[1] == "MELON"
            ):
                total += int(order[2])

        return total


    df_res["melon_sell"] = market_actions.apply(
        extract_melon_sell
    )


    # 過去特徴量
    for k in lookbacks:

        # kターン前のMELON価格
        price_k_ago = (
            df_res
            .groupby("game_id")["current_price"]
            .shift(k)
        )

        # kターン前のMELON市場在庫
        stock_k_ago = (
            df_res
            .groupby("game_id")["market_stock"]
            .shift(k)
        )

        # 過去kターンの価格変化
        df_res[f"Past_price_changes_{k}"] = (
            df_res["current_price"]
            - price_k_ago
        )

        # 過去kターンの市場在庫変化
        df_res[f"Past_inv_changes_{k}"] = (
            df_res["market_stock"]
            - stock_k_ago
        )

        # 過去kターンの自分のMELON SELL合計
        df_res[f"Past_SELL_{k}"] = (
            df_res
            .groupby("game_id")["melon_sell"]
            .transform(
                lambda s:
                s.shift(1)
                .rolling(
                    window=k,
                    min_periods=k
                )
                .sum()
            )
        )

    # 未来kターンのMELON最高価格
    for k in lookbacks:

        df_res[f"future_max_price_{k}"] = (
            df_res
            .groupby("game_id")["current_price"]
            .transform(
                lambda s:
                s.shift(-1)
                .iloc[::-1]
                .rolling(
                    window=k,
                    min_periods=k
                )
                .max()
                .iloc[::-1]
            )
        )


    df_res = df_res.drop(columns=["day",
                                  "money",
                                  "hands_n",
                                  "hires_today",
                                  "unlocked_quadrants",
                                  "tiles",
                                  "private_shed",
                                  "private_seeds",
                                  "market_inv",
                                  "market_prices",
                                  "market_action",
                                  "id",
                                  "melon_sell"

                                  ])
    df_res = df_res.dropna().reset_index(drop=True) #欠損値削除

    return df_res


In [ ]:
df_12 = df_cl(df_all, lookbacks=(12,))

In [ ]:
df_12.columns

Index(['step', 'game_id', 'amount_in_shed', 'current_price', 'market_stock',
       'Past_price_changes_12', 'Past_inv_changes_12', 'Past_SELL_12',
       'future_max_price_12'],
      dtype='object')

In [ ]:
print(df_12.isna().sum())
print(df_24.isna().sum())
print(df_48.isna().sum())

step                     0
game_id                  0
amount_in_shed           0
current_price            0
market_stock             0
Past_price_changes_12    0
Past_inv_changes_12      0
Past_SELL_12             0
future_max_price_12      0
dtype: int64
step                     0
game_id                  0
amount_in_shed           0
current_price            0
market_stock             0
Past_price_changes_24    0
Past_inv_changes_24      0
Past_SELL_24             0
future_max_price_24      0
dtype: int64
step                     0
game_id                  0
amount_in_shed           0
current_price            0
market_stock             0
Past_price_changes_48    0
Past_inv_changes_48      0
Past_SELL_48             0
future_max_price_48      0
dtype: int64


#

# 今後12ターン以内のMELON最高価格

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor
import numpy as np
import pandas as pd




df_12 = df_cl(df_all, lookbacks=(12,))

game_ids = df_12["game_id"].unique()

train_ids, test_ids = train_test_split(
    game_ids,
    test_size=0.2,
    random_state=42
)

def split_xy(df, train_ids, test_ids, k):

    train_df = df[df["game_id"].isin(train_ids)].copy()
    test_df = df[df["game_id"].isin(test_ids)].copy()

    target = f"future_max_price_{k}"

    x_train = train_df.drop(
        columns=["game_id", target]
    )

    y_train = train_df[target]


    x_test = test_df.drop(
        columns=["game_id", target]
    )

    y_test = test_df[target]

    return x_train, x_test, y_train, y_test



x_train_12, x_test_12, y_train_12, y_test_12 = split_xy(
    df_12, train_ids, test_ids, 12
)


#---------------------
# DecisionTree
#---------------------

dt = DecisionTreeRegressor(
    random_state=42
)

dt.fit(x_train_12, y_train_12)
pred_dt = dt.predict(x_test_12)

mae_dt = mean_absolute_error(y_test_12, pred_dt)
rmse_dt = np.sqrt(mean_squared_error(y_test_12, pred_dt))

print("DecisionTree")
print("MAE :", mae_dt)
print("RMSE :", rmse_dt)
print("")

#---------------------
#GBM
#---------------------

gbm = GradientBoostingRegressor(
    random_state=42
)

gbm.fit(x_train_12, y_train_12)

pred_gbm = gbm.predict(x_test_12)

mae_gbm = mean_absolute_error(y_test_12, pred_gbm)
rmse_gbm = np.sqrt(mean_squared_error(y_test_12, pred_gbm))

print("GBM")
print("MAE :", mae_gbm)
print("RMSE:", rmse_gbm)
print("")

#---------------------
# XGBoost
#---------------------

xgb = XGBRegressor(
    random_state=42
)

xgb.fit(x_train_12, y_train_12)

pred_xgb = xgb.predict(x_test_12)

mae_xgb = mean_absolute_error(y_test_12, pred_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_test_12, pred_xgb))

print("XGBoost")
print("MAE :", mae_xgb)
print("RMSE:", rmse_xgb)
print("")

DecisionTree
MAE : 0.048466176621727586
RMSE : 0.8248411469721132

GBM
MAE : 0.40869050674762836
RMSE: 1.2618446681953153

XGBoost
MAE : 0.07840405687746874
RMSE: 0.8423990014380353



# 結果



| 予測範囲  | MAE最良            |       MAE | RMSE最良           |      RMSE |
| ----- | ---------------- | --------: | ---------------- | --------: |
| 12ターン | **DecisionTree** | **0.048** | **DecisionTree** | **0.825** |
| 24ターン | **DecisionTree** | **0.108** | GBM              | **1.084** |
| 48ターン | **DecisionTree** | **0.109** | GBM              | **1.128** |


* MAE = 平均で何ドル外したか
* RMSE = どれくらいズレたか

TrainデータでDecisionTreeを学習し、

Testデータに対して、「今後kターン以内のMELON最高価格」を予測しました。




In [ ]:
!pip install m2cgen

import m2cgen as m2c

code = m2c.export_to_python(dt)
#print(code)  深すぎ！！！！

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.2/92.2 kB 3.4 MB/s eta 0:00:00



```py
def score(input):
    if input[3] <= 10092.0:
        if input[0] <= 257.5:
            if input[0] <= 255.5:
                if input[2] <= 267.5:
                    if input[3] <= 9995.5:
                        if input[3] <= 9994.5:
                            var0 = 269.0
                        else:
                            if input[0] <= 185.5:
                                var0 = 268.0
                            else:
                                if input[0] <= 253.5:
                                    if input[0] <= 252.5:
                                        .
                                        .
                                        .
                                        .
                                        .
                                        .
                                        .
                                        .
                                        .
```

まだまだ続きました。
とても出力出来ません。
なので下記に木の深さを"3"に設定した場合の実装例を書きました。

# 実装:木の深さ3の場合(max_depth=3)


```
DecisionTree
MAE : 3.175605219342661
RMSE : 5.128552085342304

```

```py
#実装例

def score(input):
    if input[3] <= 10096.5:
        if input[0] <= 257.5:
            if input[2] <= 265.0:
                var0 = 262.30210213139543
            else:
                var0 = 269.14311998948614
        else:
            if input[2] <= 180.0:
                var0 = 169.52286374133948
            else:
                var0 = 192.71375464684016
    else:
        if input[0] <= 497.5:
            if input[3] <= 10127.5:
                var0 = 95.84500378501136
            else:
                var0 = 82.26066931619805
        else:
            if input[3] <= 10154.5:
                var0 = 19.437829958238122
            else:
                var0 = 6.73007806147341
    return var0


#今後12ターン以内のMELON最高価格:(木の深さ３)

pred_price = score([
    step,
    amount_in_shed,
    current_price,
    market_stock,
    Past_price_changes_12,
    Past_inv_changes_12,
    Past_SELL_12,
])


if pred_price > current_price:
    # HOLD
else:
    # SELL


```

この場合、実際に使用されてる特徴量↓
```
input[0]  # step
input[2]  # current_price
input[3]  # market_stock
```



# 終わりに

ゲームのルール上
このコードを実装できるかどうかはまだ不確定。